# Day 0: Python warm-up

Plain Python first, then numpy, then pandas. Small loan-book data so you can check answers by eye.

**Rules**
- Type every `TODO` yourself. No AI, no copy-paste from elsewhere. Googling syntax is fine.
- Each exercise has a CHECK cell. Run it. If it errors, fix *your* code.
- Stuck for 15 minutes? Look at that one exercise in the solutions notebook, then close it and **retype it from memory**.
- Speed doesn't matter today. Writing it yourself does.

## Part A: plain Python

### A1. Variables and arithmetic
A loan has a balance of £12,000 and an annual interest rate of 6.5%. Create `balance`, `rate` (as a decimal) and `monthly_interest` (one month's interest).

In [ ]:

balance = 12000
rate = 0.065
monthly_interest = balance * rate/12
print(monthly_interest)

65.0


In [5]:
# CHECK
assert round(monthly_interest, 2) == 65.0
print('A1 passed')

A1 passed


### A2. if / elif / else
Write `arrears_bucket(dpd)` that takes days past due and returns:
- `'Current'` if 0
- `'1-29'`, `'30-59'`, `'60-89'` for those ranges
- `'90+'` for 90 or more

In [12]:
def arrears_bucket(dpd):
    if dpd == 0:
        return 'Current'
    elif dpd < 30:
        return '1-29'
    elif dpd < 60:
        return '30-59'
    elif dpd < 90:
        return '60-89'
    else:
        return '90+'

In [13]:
arrears_bucket(0)

'Current'

In [14]:
# CHECK
assert arrears_bucket(0) == 'Current'
assert arrears_bucket(29) == '1-29'
assert arrears_bucket(30) == '30-59'
assert arrears_bucket(89) == '60-89'
assert arrears_bucket(90) == '90+'
assert arrears_bucket(400) == '90+'
print('A2 passed')

A2 passed


Notice the checks test the **edges** (29, 30, 89, 90). That's exactly where SAS-to-Python breaks hide. Get used to testing boundaries.

### A3. Lists and for loops
Using a `for` loop (not `sum` or a comprehension), count how many loans in `dpd_list` are 90+ days past due. Store the count in `n_defaults`.

In [29]:
dpd_list = [0, 15, 45, 90, 120, 0, 30, 95, 0, 60]


n_defaults = 0

for i in dpd_list:
    if i >= 90:
        n_defaults = n_defaults +1
        print(n_defaults)
    else:
        n_defaults =n_defaults

print(n_defaults)

1
2
3
3


In [30]:
# CHECK
assert n_defaults == 3
print('A3 passed')

A3 passed


### A4. List comprehensions
Do it in one line this time: make `default_flags`, a list of 1s and 0s (1 if 90+), then `default_rate` = share of loans in default.

In [31]:
dpd_list = [0, 15, 45, 90, 120, 0, 30, 95, 0, 60]

In [35]:
default_flags = [1 if d >= 90 else 0 for d in dpd_list]
default_rate = sum(default_flags) / len(default_flags)
print(default_flags, default_rate)

[0, 0, 0, 1, 1, 0, 0, 1, 0, 0] 0.3


In [36]:
# CHECK
assert default_flags == [0, 0, 0, 1, 1, 0, 0, 1, 0, 0]
assert default_rate == 0.3
print('A4 passed')

A4 passed


Also use your `arrears_bucket` function inside a comprehension to make `buckets`, a list of bucket labels for `dpd_list`.

In [37]:
def arrears_bucket(dpd):
    if dpd == 0:
        return 'Current'
    elif dpd < 30:
        return '1-29'
    elif dpd < 60:
        return '30-59'
    elif dpd < 90:
        return '60-89'
    else:
        return '90+'

In [38]:
dpd_list = [0, 15, 45, 90, 120, 0, 30, 95, 0, 60]

In [39]:
# TODO




buckets = [arrears_bucket(d) for d in dpd_list]
print(buckets)

['Current', '1-29', '30-59', '90+', '90+', 'Current', '30-59', '90+', 'Current', '60-89']


In [40]:
# CHECK
assert buckets[3] == '90+' and buckets[2] == '30-59' and buckets.count('Current') == 3
print('A4b passed')

A4b passed


### A5. Dictionaries
1. Read the balance out of `loan`.
2. Add a new key `'ltv'` = balance / property_value.
3. Loop over the dictionary and print each key and value on its own line.

In [41]:
loan = {'loan_id': 'L001', 'balance': 150000, 'property_value': 200000, 'region': 'North West'}

# TODO
bal = ...
# add ltv

# loop and print


In [46]:
loan = {'loan_id': 'L001', 'balance': 150000, 'property_value': 200000, 'region': 'North West'}

bal = loan['balance']
loan['ltv'] = loan['balance'] / loan['property_value']

for key, value in loan.items():
    print(key, value)

loan_id L001
balance 150000
property_value 200000
region North West
ltv 0.75


In [47]:
# CHECK
assert bal == 150000
assert loan['ltv'] == 0.75
print('A5 passed')

A5 passed


### A6. Functions with default arguments
Write `monthly_payment(principal, annual_rate, years=25)` for a repayment mortgage:

$$\text{payment} = P \cdot \frac{r}{1 - (1 + r)^{-n}}$$

where $r$ = annual_rate / 12 and $n$ = years × 12.

In [48]:
# TODO
def monthly_payment(principal, annual_rate, years=25):
    r=annual_rate/12
    n=years*12
    upper = r
    lower = (1-(1+r)**(-n))
    payment=principal * upper/lower

    return payment

In [49]:
# CHECK
assert round(monthly_payment(200000, 0.05), 2) == 1169.18
assert round(monthly_payment(200000, 0.05, years=30), 2) == 1073.64
print('A6 passed')

A6 passed


### A7. Handling missing values
Real data has gaps. Update the idea from A2: write `safe_bucket(dpd)` that returns `'Unknown'` if `dpd` is `None`, otherwise calls `arrears_bucket`.

*Why this matters:* in SAS, a missing value counts as **smaller than any number**, so `if dpd < 30` is TRUE for a missing dpd. In Python, `None < 30` raises an error. Same data, different behaviour. That's a reconciliation break.

In [51]:
dpd_list = [0, 15, 45, 90, 120, 0, 30, 95, 0, 60]

In [52]:
def arrears_bucket(dpd):
    if dpd == 0:
        return 'Current'
    elif dpd < 30:
        return '1-29'
    elif dpd < 60:
        return '30-59'
    elif dpd < 90:
        return '60-89'
    else:
        return '90+'

In [ ]:

def safe_bucket(dpd):
    if dpd ==

In [ ]:
# CHECK
assert safe_bucket(None) == 'Unknown'
assert safe_bucket(45) == '30-59'
print('A7 passed')

## Part B: numpy

### B1. Arrays
`pds` holds predicted PDs for 8 loans. Using numpy (no loops):
1. `mean_pd`: the average PD
2. `n_high`: how many PDs are above 5%
3. `risk_label`: an array saying `'High'` where PD > 5%, else `'Low'` (hint: `np.where`)

In [70]:
import numpy as np
pds = np.array([0.01, 0.03, 0.08, 0.002, 0.15, 0.04, 0.06, 0.005])

# TODO
mean_pd = sum(pds/len(pds))
n_high = (pds>0.05).sum()
risk_label = np.where(pds > 0.05, 'High', 'Low')
print(mean_pd, n_high, risk_label)

0.047125 3 ['Low' 'Low' 'High' 'Low' 'High' 'Low' 'High' 'Low']


In [71]:
# CHECK
assert np.isclose(mean_pd, 0.047125)
assert n_high == 3
assert list(risk_label) == ['Low','Low','High','Low','High','Low','High','Low']
print('B1 passed')

B1 passed


**Think:** `(pds > 0.05).sum()` works because True counts as 1. Same trick you used for default_rate in A4.

## Part C: pandas

### C0. The data (given)

In [72]:
import pandas as pd
loans = pd.DataFrame({
    'loan_id':        ['L01','L02','L03','L04','L05','L06','L07','L08','L09','L10'],
    'region':         ['North West','London','North West','Scotland','London','Scotland','North West','London','Scotland','North West'],
    'balance':        [150000, 320000, 95000, 120000, 410000, 80000, 175000, 290000, 60000, 130000],
    'property_value': [200000, 400000, 160000, 150000, 450000, 140000, 190000, 380000, 110000, 210000],
    'dpd':            [0, 0, 45, 0, 95, 0, 120, 30, 0, 0],
})
loans

,loan_id,region,balance,property_value,dpd
0,L01,North West,150000,200000,0
1,L02,London,320000,400000,0
2,L03,North West,95000,160000,45
3,L04,Scotland,120000,150000,0
4,L05,London,410000,450000,95
5,L06,Scotland,80000,140000,0
6,L07,North West,175000,190000,120
7,L08,London,290000,380000,30
8,L09,Scotland,60000,110000,0
9,L10,North West,130000,210000,0


### C1. Selecting
1. `ids_and_dpd`: a DataFrame with only `loan_id` and `dpd`
2. `in_arrears`: only the rows where dpd > 0
3. `london_big`: London loans with balance over £300k (two conditions: use `&` and brackets)

In [81]:
# TODO
ids_and_dpd = loans[['loan_id','dpd']]
in_arrears = loans[loans['dpd'] > 0]
london_big = loans[(loans['balance'] > 300000) & (loans['region'] == 'London')]
ids_and_dpd

,loan_id,dpd
0,L01,0
1,L02,0
2,L03,45
3,L04,0
4,L05,95
5,L06,0
6,L07,120
7,L08,30
8,L09,0
9,L10,0


In [82]:
# CHECK
assert list(ids_and_dpd.columns) == ['loan_id', 'dpd']
assert len(in_arrears) == 4
assert list(london_big['loan_id']) == ['L02', 'L05']
print('C1 passed')

C1 passed


### C2. New columns
Add to `loans`:
1. `ltv` = balance / property_value
2. `default_flag` = 1 if dpd >= 90 else 0 (as integers, no loop: compare, then `.astype(int)`)
3. `bucket` = your `arrears_bucket` applied to dpd (hint: `.apply`)

In [90]:

loans['ltv'] = loans['balance']/loans['property_value']
loans['default_flag'] = (loans['dpd'] >= 90).astype(int)
loans['bucket']=loans['dpd'].apply(arrears_bucket)
loans

,loan_id,region,balance,property_value,dpd,ltv,default_flag,bucket
0,L01,North West,150000,200000,0,0.750000,0,Current
1,L02,London,320000,400000,0,0.800000,0,Current
2,L03,North West,95000,160000,45,0.593750,0,30-59
3,L04,Scotland,120000,150000,0,0.800000,0,Current
4,L05,London,410000,450000,95,0.911111,1,90+
5,L06,Scotland,80000,140000,0,0.571429,0,Current
6,L07,North West,175000,190000,120,0.921053,1,90+
7,L08,London,290000,380000,30,0.763158,0,30-59
8,L09,Scotland,60000,110000,0,0.545455,0,Current
9,L10,North West,130000,210000,0,0.619048,0,Current


In [91]:
# CHECK
assert loans.loc[loans['loan_id']=='L04', 'ltv'].item() == 0.8
assert loans['default_flag'].sum() == 2
assert loans['default_flag'].dtype.kind == 'i'
assert loans['bucket'].value_counts()['Current'] == 6
print('C2 passed')

C2 passed


### C3. Sorting and counting
1. `top3`: the 3 loans with the highest LTV, highest first
2. `bucket_counts`: how many loans in each bucket (hint: `value_counts`)

In [92]:
top3 = loans.sort_values('ltv', ascending=False).head(3)
bucket_counts = loans['bucket'].value_counts()
print(top3[['loan_id','ltv']])
print(bucket_counts)

  loan_id       ltv
6     L07  0.921053
4     L05  0.911111
1     L02  0.800000
bucket
Current    6
30-59      2
90+        2
Name: count, dtype: int64


In [93]:
# CHECK
assert list(top3['loan_id'])[:2] == ['L07', 'L05'] and top3['loan_id'].iloc[2] in ('L02', 'L04')
assert bucket_counts['90+'] == 2
print('C3 passed')

C3 passed


### C4. groupby
Make `by_region`: one row per region with three columns: `n_loans`, `total_balance`, `default_rate`.

*Hint: `.groupby('region').agg(n_loans=('loan_id', 'count'), ...)`*

In [96]:
by_region = loans.groupby('region').agg(
    n_loans=('loan_id', 'count'),
    total_balance=('balance', 'sum'),
    default_rate=('default_flag', 'mean'),
)
by_region

,n_loans,total_balance,default_rate
region,,,
London,3,1020000,0.333333
North West,4,550000,0.250000
Scotland,3,260000,0.000000


In [97]:
# CHECK
assert by_region.loc['North West', 'n_loans'] == 4
assert by_region.loc['London', 'total_balance'] == 1020000
assert round(by_region.loc['London', 'default_rate'], 4) == 0.3333
print('C4 passed')

C4 passed


### C5. Merging: your first reconciliation
The "old system" and "new system" each scored the loans. Your job:
1. Merge them on `loan_id` with an **outer** join and `indicator=True`, call it `recon`.
2. `missing`: rows where a loan is in only one system (indicator isn't `'both'`).
3. `breaks`: loans in both where the PDs differ by more than 0.0001 (hint: `.abs()`).

In [ ]:
old = pd.DataFrame({'loan_id': ['L01','L02','L03','L04','L05','L06'],
                    'pd_old':  [0.010, 0.020, 0.150, 0.030, 0.400, 0.015]})
new = pd.DataFrame({'loan_id': ['L01','L02','L03','L05','L06','L07'],
                    'pd_new':  [0.010, 0.020, 0.135, 0.400, 0.01500001, 0.500]})

recon = old.merge(new, on='loan_id', how='outer', indicator=True)
missing = recon[recon['_merge'] != 'both']
both = recon[recon['_merge'] == 'both']
breaks = both[(both['pd_old'] - both['pd_new']).abs() > 0.0001]
print(missing[['loan_id', '_merge']])
print(breaks)

  loan_id      _merge
3     L04   left_only
6     L07  right_only
  loan_id  pd_old  pd_new _merge
2     L03    0.15   0.135   both


In [105]:
breaks

,loan_id,pd_old,pd_new,_merge
2,L03,0.15,0.135,both


In [103]:
missing

,loan_id,pd_old,pd_new,_merge
3,L04,0.03,NaN,left_only
6,L07,NaN,0.5,right_only


In [104]:
both

,loan_id,pd_old,pd_new,_merge
0,L01,0.010,0.010,both
1,L02,0.020,0.020,both
2,L03,0.150,0.135,both
4,L05,0.400,0.400,both
5,L06,0.015,0.015,both


In [101]:
new

,loan_id,pd_new
0,L01,0.010
1,L02,0.020
2,L03,0.135
3,L05,0.400
4,L06,0.015
5,L07,0.500


In [100]:
old

,loan_id,pd_old
0,L01,0.010
1,L02,0.020
2,L03,0.150
3,L04,0.030
4,L05,0.400
5,L06,0.015


In [102]:
recon

,loan_id,pd_old,pd_new,_merge
0,L01,0.010,0.010,both
1,L02,0.020,0.020,both
2,L03,0.150,0.135,both
3,L04,0.030,NaN,left_only
4,L05,0.400,0.400,both
5,L06,0.015,0.015,both
6,L07,NaN,0.500,right_only


In [ ]:
# CHECK
assert sorted(missing['loan_id']) == ['L04', 'L07']
assert list(breaks['loan_id']) == ['L03']
print('C5 passed')

**Say out loud:** L06 differs by 0.00000001 but isn't a break. Why is that the right call, and who decides the tolerance?

---
**Done?** Tell me which exercises you got without peeking and which you had to look up. That tells me whether to go to Day 1 next or do one more warm-up round first.